# HuggingFace Model Migration

This notebook shows how to migrate your Lightning checkpoint to a HuggingFace-compatible model format that decouples code from checkpoints.

## Why Migrate?

1. **Code-Checkpoint Decoupling**: Load models without maintaining source code
2. **Easy Sharing**: Push to HuggingFace Hub with one command
3. **Standard API**: Use `AutoModel.from_pretrained()` just like any other HF model
4. **Future-Proof**: Update code without affecting model loading

In [1]:
import os
os.chdir("../")

import sys
import torch
import json
from pathlib import Path

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


## Step 1: Setup - Load Your Current Checkpoint

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Point to your checkpoint
# ckpt_path = "logs/train/runs/2026-01-29_13-44-38/checkpoints/last.ckpt"
# ckpt_path = "checkpoints/wong_sngp_resnet18/model.ckpt"
# ckpt_path = "checkpoints/acevedo_baseline_resnet18/model.ckpt"
ckpt_path = "checkpoints/wong_baseline_resnet18/model.ckpt"
checkpoint = torch.load(ckpt_path, map_location=device, weights_only=False)
print(f"Checkpoint keys: {checkpoint.keys()}")

Using device: cuda
Checkpoint keys: dict_keys(['epoch', 'global_step', 'pytorch-lightning_version', 'state_dict', 'loops', 'callbacks', 'optimizer_states', 'lr_schedulers', 'hparams_name', 'hyper_parameters', 'datamodule_hparams_name', 'datamodule_hyper_parameters'])


/home/wisc/maheswararao/code/lightning-hydra-template/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Inspect state dict structure
state_dict = checkpoint['state_dict']
print(f"Number of parameters: {len(state_dict)}")
print(f"First 5 keys:\n{list(state_dict.keys())[:5]}")

Number of parameters: 122
First 5 keys:
['net.feature_extractor.conv1.weight', 'net.feature_extractor.bn1.weight', 'net.feature_extractor.bn1.bias', 'net.feature_extractor.bn1.running_mean', 'net.feature_extractor.bn1.running_var']


## Step 2: Extract Model Parameters from Checkpoint

The Lightning checkpoint has keys like `net.feature_extractor.layer1...` because they're wrapped. We need to remove the `net.` prefix.

In [5]:
def extract_state_dict(checkpoint, device):
    """Extract and remap state dict from Lightning checkpoint."""
    state_dict = checkpoint['state_dict']
    
    # Remove 'net.' prefix (Lightning module wrapper)
    remapped_state_dict = {}
    for key, value in state_dict.items():
        # Remove 'net.' prefix
        new_key = key.replace('net.', '', 1)
        remapped_state_dict[new_key] = value
    
    return remapped_state_dict

# Extract the clean state dict
clean_state_dict = extract_state_dict(checkpoint, device)
print(f"Remapped state dict keys (first 5):\n{list(clean_state_dict.keys())[:5]}")

Remapped state dict keys (first 5):
['feature_extractor.conv1.weight', 'feature_extractor.bn1.weight', 'feature_extractor.bn1.bias', 'feature_extractor.bn1.running_mean', 'feature_extractor.bn1.running_var']


## Step 3: Create HuggingFace Model Directory Structure

In [6]:
# Configuration for your model
#acevedo
# model_config = {
#     "arch": "resnet18",           # Match your architecture
#     "num_classes": 8,              # Match your num_classes
#     "dropout_p": 0.5,              # Match your dropout
#     "pretrained": False,           # No need to re-pretrain
# }

#wong

model_config = {
    "arch": "resnet18",           # Match your architecture
    "num_classes": 4,              # Match your num_classes
    "dropout_p": 0.,              # Match your dropout
    "pretrained": False,           # No need to re-pretrain
}


# Create output directory
# hf_model_dir = Path("./checkpoints/acevedo_baseline_resnet18/hf_checkpoint")
hf_model_dir = Path("./checkpoints/wong_baseline_resnet18/hf_checkpoint")
hf_model_dir.mkdir(parents=True, exist_ok=True)

print(f"Created directory: {hf_model_dir}")
print(f"Model config: {model_config}")

Created directory: checkpoints/wong_baseline_resnet18/hf_checkpoint
Model config: {'arch': 'resnet18', 'num_classes': 4, 'dropout_p': 0.2, 'pretrained': False}


In [7]:
# Create config.json - This tells HF how to load the model
config_dict = {
    "architectures": ["BaselineClassifierForImageClassification"],
    "model_type": "baseline_classifier",
    "auto_map": {
        "AutoConfig": "hf_model.BaselineClassifierConfig",
        "AutoModel": "hf_model.BaselineClassifierForImageClassification",
    },
    **model_config,
}

config_path = hf_model_dir / "config.json"
with open(config_path, "w") as f:
    json.dump(config_dict, f, indent=2)

print(f"✓ Saved config.json")
print(json.dumps(config_dict, indent=2))

✓ Saved config.json
{
  "architectures": [
    "BaselineClassifierForImageClassification"
  ],
  "model_type": "baseline_classifier",
  "auto_map": {
    "AutoConfig": "hf_model.BaselineClassifierConfig",
    "AutoModel": "hf_model.BaselineClassifierForImageClassification"
  },
  "arch": "resnet18",
  "num_classes": 4,
  "dropout_p": 0.2,
  "pretrained": false
}


In [8]:
# Save model weights
weights_path = hf_model_dir / "pytorch_model.bin"
torch.save(clean_state_dict, weights_path)
print(f"✓ Saved pytorch_model.bin ({weights_path.stat().st_size / 1e6:.1f} MB)")

✓ Saved pytorch_model.bin (44.8 MB)


## Step 4: Save the Model Code (`hf_model.py`)

This is the key step for decoupling! We save the model code in the checkpoint directory.

In [9]:
# Copy the HF model code
import shutil
from src.hf_model import BaselineClassifierConfig, BaselineClassifier, BaselineClassifierForImageClassification

# The hf_model.py is already in src/
hf_model_src = Path("src/hf_model.py")
hf_model_dst = hf_model_dir / "hf_model.py"
shutil.copy(hf_model_src, hf_model_dst)

print(f"✓ Copied hf_model.py to checkpoint directory")
print(f"  Source: {hf_model_src}")
print(f"  Dest: {hf_model_dst}")

✓ Copied hf_model.py to checkpoint directory
  Source: src/hf_model.py
  Dest: checkpoints/wong_baseline_resnet18/hf_checkpoint/hf_model.py


## Step 5: Test Loading the Model Locally

In [10]:
from transformers import AutoConfig, AutoModel

# Load config
config = AutoConfig.from_pretrained(str(hf_model_dir), trust_remote_code=True)
print(f"✓ Loaded config: {config}")

✓ Loaded config: BaselineClassifierConfig {
  "arch": "resnet18",
  "architectures": [
    "BaselineClassifierForImageClassification"
  ],
  "auto_map": {
    "AutoConfig": "hf_model.BaselineClassifierConfig",
    "AutoModel": "hf_model.BaselineClassifierForImageClassification"
  },
  "dropout_p": 0.2,
  "model_type": "baseline_classifier",
  "num_classes": 4,
  "pretrained": false,
  "transformers_version": "4.57.3"
}



In [11]:
# Load model using HuggingFace API (without source code!)
model = AutoModel.from_pretrained(str(hf_model_dir), trust_remote_code=True)
model = model.to(device)
model.eval()

print(f"✓ Model loaded successfully!")
print(f"  Type: {type(model)}")
print(f"  Device: {next(model.parameters()).device}")
print(f"  Model structure:\n{model}")

✓ Model loaded successfully!
  Type: <class 'transformers_modules.hf_checkpoint.hf_model.BaselineClassifierForImageClassification'>
  Device: cuda:0
  Model structure:
BaselineClassifierForImageClassification(
  (model): BaselineClassifier(
    (feature_extractor): ResNet(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): BasicBlock(
          (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (relu): ReLU(inplace=True)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): BatchNorm2d(64, ep

In [12]:
# Test inference with dummy input
test_input = torch.randn(2, 3, 224, 224).to(device)

with torch.no_grad():
    output = model(test_input)

print(f"✓ Inference successful!")
print(f"  Input shape: {test_input.shape}")
print(f"  Output: {output}")
if isinstance(output, dict):
    print(f"  Logits shape: {output['logits'].shape}")
else:
    print(f"  Output shape: {output.shape}")

✓ Inference successful!
  Input shape: torch.Size([2, 3, 224, 224])
  Output: {'logits': tensor([[-28.2298, -11.1146,  -3.5716,  15.1821],
        [-30.3750,  -9.7913,  -3.8794,  15.5005]], device='cuda:0')}
  Logits shape: torch.Size([2, 4])


## Step 6: Verify Model Outputs Match Original

Load the original model to ensure our migrated model produces identical outputs.

In [45]:
# Compare outputs
with torch.no_grad():
    original_out = original_model(test_input)
    
    hf_out = model(test_input)
    if isinstance(hf_out, dict):
        hf_out = hf_out['logits']

# Check if outputs match
max_diff = (original_out - hf_out).abs().max().item()
print(f"✓ Output comparison:")
print(f"  Max difference: {max_diff:.2e}")
print(f"  Match: {torch.allclose(original_out, hf_out, atol=1e-5)}")

if torch.allclose(original_out, hf_out, atol=1e-5):
    print("  ✓ Outputs are numerically identical!")
else:
    print("  ⚠ Outputs differ - check model loading")

✓ Output comparison:
  Max difference: 0.00e+00
  Match: True
  ✓ Outputs are numerically identical!


## Step 7: (Optional) Push to HuggingFace Hub

In [ ]:
# Install huggingface_hub if needed
# import subprocess
# subprocess.run(["pip", "install", "huggingface_hub"], check=True)

In [21]:
try:
    from huggingface_hub import upload_folder, get_token
    
    # Check if logged in
    token = get_token()
    if token is None:
        print("⚠ Not logged into HuggingFace Hub")
        print("  Run: huggingface-cli login")
    else:
        print("✓ Logged into HuggingFace Hub")
        print("  Ready to push!")
        
except ImportError:
    print("⚠ huggingface_hub not installed")
    print("  Install with: pip install huggingface_hub")

✓ Logged into HuggingFace Hub
  Ready to push!


In [ ]:
"""
PUSH TO HUB EXAMPLE - Push to specific model variant:

from huggingface_hub import upload_folder

# Option 1: Push to root (main model)
repo_id = "nirschl-lab/sngp-models"
upload_folder(
    repo_id=repo_id,
    folder_path=str(hf_model_dir),
    repo_type="model",
)

# Option 2: Push to a subdirectory (model variant)
# This keeps multiple models organized in one repo
upload_folder(
    repo_id=repo_id,
    folder_path=str(hf_model_dir),
    path_in_repo="resnet18-v1",  # Creates: nirschl-lab/sngp-models/resnet18-v1/
    repo_type="model",
)

print(f"✓ Model pushed to: https://huggingface.co/{repo_id}")
"""

In [46]:
# Option 2: Push to a subdirectory (model variant)
# This keeps multiple models organized in one repo
repo_id = "nirschl-lab/sngp-models"
upload_folder(
    repo_id=repo_id,
    folder_path=str(hf_model_dir),
    path_in_repo="wong_baseline_resnet18",  # Creates: nirschl-lab/sngp-models/wong_baseline_resnet18/
    repo_type="model",
)

print(f"✓ Model pushed to: https://huggingface.co/{repo_id}")

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

Processing Files (0 / 1)                :   2%|▏         | 1.12MB / 44.8MB, 1.86MB/s  
Processing Files (0 / 1)                :  57%|█████▋    | 25.7MB / 44.8MB, 32.1MB/s  
Processing Files (0 / 1)                : 100%|█████████▉| 44.7MB / 44.8MB, 44.7MB/s  


Processing Files (1 / 1)                : 100%|██████████| 44.8MB / 44.8MB, 28.0MB/s  

Processing Files (1 / 1)                : 100%|██████████| 44.8MB / 44.8MB, 24.9MB/s  
New Data Upload                         : 100%|██████████| 44.8MB / 44.8MB, 24.9MB/s  
  ...t18/hf_checkpoint/pytorch_model.bin: 100%|██████████| 44.8MB / 44.8MB            


✓ Model pushed to: https://huggingface.co/nirschl-lab/sngp-models


## Organizing Multiple Models in Your Hub Repo

You can keep multiple model variants organized in your single repository using subdirectories. This is perfect for:
- Different architectures (resnet18, resnet50, vit_b_16)
- Different training datasets/versions
- Different configurations (dropout rates, etc.)

**Repo structure:**
```
nirschl-lab/sngp-models/
├── resnet18-v1/
│   ├── config.json
│   ├── pytorch_model.bin
│   └── hf_model.py
├── resnet50-v1/
│   ├── config.json
│   ├── pytorch_model.bin
│   └── hf_model.py
├── vit_b_16-v1/
│   ├── config.json
│   ├── pytorch_model.bin
│   └── hf_model.py
└── README.md (describes all variants)
```

In [5]:
# Check if you can access the model directly
from huggingface_hub import model_info

try:
    info = model_info("nirschl-lab/sngp-models")
    print(info)
except Exception as e:
    print(f"Model not found: {e}")

ModelInfo(id='nirschl-lab/sngp-models', author='nirschl-lab', sha='162c17cbd71cf4541b704ebaf91fe6b41cfe3ec7', created_at=datetime.datetime(2026, 2, 2, 18, 8, 56, tzinfo=datetime.timezone.utc), last_modified=datetime.datetime(2026, 2, 3, 20, 10, 29, tzinfo=datetime.timezone.utc), private=True, disabled=False, downloads=0, downloads_all_time=None, gated=False, gguf=None, inference=None, inference_provider_mapping=None, likes=0, library_name=None, tags=['license:mit', 'region:us'], pipeline_tag=None, mask_token=None, card_data={'base_model': None, 'datasets': None, 'eval_results': None, 'language': None, 'library_name': None, 'license': 'mit', 'license_name': None, 'license_link': None, 'metrics': None, 'model_name': None, 'pipeline_tag': None, 'tags': None}, widget_data=None, model_index=None, config={}, transformers_info=None, trending_score=None, siblings=[RepoSibling(rfilename='.gitattributes', size=None, blob_id=None, lfs=None), RepoSibling(rfilename='README.md', size=None, blob_id=N

In [47]:
# Load different model variants from your repo
from transformers import AutoModel
from huggingface_hub import snapshot_download
from pathlib import Path

repo_id = "nirschl-lab/sngp-models"
variant_name = "wong_baseline_resnet18"

try:
    # Download the specific subfolder
    cache_dir = snapshot_download(
        repo_id=repo_id,
        allow_patterns=f"{variant_name}/*",
        cache_dir="./hf_models_cache"
    )
    
    # Find the subfolder path (it's in HF's cache structure)
    subfolder_path = Path(cache_dir) / variant_name
    
    print(f"✓ Downloaded model to: {subfolder_path}")
    print(f"  Files: {list(subfolder_path.glob('*'))}")
    
    # Load directly from the subfolder path
    model = AutoModel.from_pretrained(
        str(subfolder_path),
        trust_remote_code=True,
    )
    model = model.to(device)
    model.eval()
    
    print("✓ Model loaded successfully!")
    print(f"  Model type: {type(model).__name__}")
    
    # Test inference
    test_input = torch.randn(1, 3, 224, 224).to(device)
    with torch.no_grad():
        output = model(test_input)
    print(f"  ✓ Inference successful!")
    if isinstance(output, dict):
        print(f"    Output shape: {output['logits'].shape}")
    else:
        print(f"    Output shape: {output.shape}")
    
except Exception as e:
    print(f"✗ Failed to load model: {e}")
    import traceback
    traceback.print_exc()

Fetching 3 files: 100%|██████████| 3/3 [00:01<00:00,  2.87it/s]

✓ Downloaded model to: hf_models_cache/models--nirschl-lab--sngp-models/snapshots/c037bd48f2ab51b7756593e119b194ef5eb1fcb3/wong_baseline_resnet18
  Files: [PosixPath('hf_models_cache/models--nirschl-lab--sngp-models/snapshots/c037bd48f2ab51b7756593e119b194ef5eb1fcb3/wong_baseline_resnet18/hf_model.py'), PosixPath('hf_models_cache/models--nirschl-lab--sngp-models/snapshots/c037bd48f2ab51b7756593e119b194ef5eb1fcb3/wong_baseline_resnet18/pytorch_model.bin'), PosixPath('hf_models_cache/models--nirschl-lab--sngp-models/snapshots/c037bd48f2ab51b7756593e119b194ef5eb1fcb3/wong_baseline_resnet18/config.json')]
✓ Model loaded successfully!
  Model type: BaselineClassifierForImageClassification
  ✓ Inference successful!
    Output shape: torch.Size([1, 4])


## Usage Examples

### 1. Load from Local Directory

```python
from transformers import AutoModel

# Load from local directory
model = AutoModel.from_pretrained("./hf_checkpoint", trust_remote_code=True)
model.eval()

# Inference
import torch
input_tensor = torch.randn(1, 3, 224, 224)
with torch.no_grad():
    output = model(input_tensor)
```

### 2. Load from HuggingFace Hub (After Pushing)

```python
from transformers import AutoModel

# Load from HuggingFace Hub
model = AutoModel.from_pretrained("your-username/my-awesome-model", trust_remote_code=True)
model.eval()

# Use exactly as before!
output = model(input_tensor)
```

### 3. No Source Code Dependency

```python
# This works WITHOUT your src/ directory!
# The model code is bundled in hf_model.py
# Only needs: torch, transformers, torchvision

model = AutoModel.from_pretrained("your-username/my-awesome-model", trust_remote_code=True)
```

## Usage Examples

### 1. Load from Local Directory

```python
from transformers import AutoModel

# Load from local directory
model = AutoModel.from_pretrained("./hf_checkpoint", trust_remote_code=True)
model.eval()

# Inference
import torch
input_tensor = torch.randn(1, 3, 224, 224)
with torch.no_grad():
    output = model(input_tensor)
```

### 2. Load from HuggingFace Hub (After Pushing)

```python
from transformers import AutoModel

# Load from HuggingFace Hub
model = AutoModel.from_pretrained("your-username/my-awesome-model", trust_remote_code=True)
model.eval()

# Use exactly as before!
output = model(input_tensor)
```

### 3. No Source Code Dependency

```python
# This works WITHOUT your src/ directory!
# The model code is bundled in hf_model.py
# Only needs: torch, transformers, torchvision

model = AutoModel.from_pretrained("your-username/my-awesome-model", trust_remote_code=True)
```

In [ ]:
"""
PUSH TO HUB EXAMPLE (uncomment to use):

from huggingface_hub import upload_folder

repo_id = "your-username/my-awesome-model"  # Change this!

upload_folder(
    repo_id=repo_id,
    folder_path=str(hf_model_dir),
    repo_type="model",
)

print(f"✓ Model pushed to: https://huggingface.co/{repo_id}")
"""

In [1]:
try:
    from huggingface_hub import upload_folder, get_token
    
    # Check if logged in
    token = get_token()
    if token is None:
        print("⚠ Not logged into HuggingFace Hub")
        print("  Run: huggingface-cli login")
    else:
        print("✓ Logged into HuggingFace Hub")
        print("  Ready to push!")
        
except ImportError:
    print("⚠ huggingface_hub not installed")
    print("  Install with: pip install huggingface_hub")

✓ Logged into HuggingFace Hub
  Ready to push!


/home/wisc/maheswararao/code/lightning-hydra-template/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Install huggingface_hub if needed
# import subprocess
# subprocess.run(["pip", "install", "huggingface_hub"], check=True)

## Step 7: (Optional) Push to HuggingFace Hub

In [ ]:
# Compare outputs
with torch.no_grad():
    original_out = original_model(test_input)
    
    hf_out = model(test_input)
    if isinstance(hf_out, dict):
        hf_out = hf_out['logits']

# Check if outputs match
max_diff = (original_out - hf_out).abs().max().item()
print(f"✓ Output comparison:")
print(f"  Max difference: {max_diff:.2e}")
print(f"  Match: {torch.allclose(original_out, hf_out, atol=1e-5)}")

if torch.allclose(original_out, hf_out, atol=1e-5):
    print("  ✓ Outputs are numerically identical!")
else:
    print("  ⚠ Outputs differ - check model loading")

In [ ]:
from src.models.baseline.baseline_models import BaselineClassifier

# Load original model
original_model = BaselineClassifier(arch="resnet18", num_classes=8, dropout_p=0.5, pretrained=False)
original_model = original_model.to(device)
original_model.eval()

# Load checkpoint into original model
original_state_dict = checkpoint['state_dict']
remapped = {}
for k, v in original_state_dict.items():
    remapped[k.replace('net.', '', 1)] = v
original_model.load_state_dict(remapped)

print("✓ Original model loaded")

## Step 6: Verify Model Outputs Match Original

Load the original model to ensure our migrated model produces identical outputs.

In [ ]:
# Test inference with dummy input
test_input = torch.randn(2, 3, 224, 224).to(device)

with torch.no_grad():
    output = model(test_input)

print(f"✓ Inference successful!")
print(f"  Input shape: {test_input.shape}")
print(f"  Output: {output}")
if isinstance(output, dict):
    print(f"  Logits shape: {output['logits'].shape}")
else:
    print(f"  Output shape: {output.shape}")

In [ ]:
# Load model using HuggingFace API (without source code!)
model = AutoModel.from_pretrained(str(hf_model_dir), trust_remote_code=True)
model = model.to(device)
model.eval()

print(f"✓ Model loaded successfully!")
print(f"  Type: {type(model)}")
print(f"  Device: {next(model.parameters()).device}")
print(f"  Model structure:\n{model}")

In [ ]:
from transformers import AutoConfig, AutoModel

# Load config
config = AutoConfig.from_pretrained(str(hf_model_dir), trust_remote_code=True)
print(f"✓ Loaded config: {config}")

## Step 5: Test Loading the Model Locally

In [ ]:
# Copy the HF model code
import shutil
from src.hf_model import BaselineClassifierConfig, BaselineClassifier, BaselineClassifierForImageClassification

# The hf_model.py is already in src/
hf_model_src = Path("src/hf_model.py")
hf_model_dst = hf_model_dir / "hf_model.py"
shutil.copy(hf_model_src, hf_model_dst)

print(f"✓ Copied hf_model.py to checkpoint directory")
print(f"  Source: {hf_model_src}")
print(f"  Dest: {hf_model_dst}")

## Step 4: Save the Model Code (`hf_model.py`)

This is the key step for decoupling! We save the model code in the checkpoint directory.

In [ ]:
# Save model weights
weights_path = hf_model_dir / "pytorch_model.bin"
torch.save(clean_state_dict, weights_path)
print(f"✓ Saved pytorch_model.bin ({weights_path.stat().st_size / 1e6:.1f} MB)")

In [ ]:
# Create config.json - This tells HF how to load the model
config_dict = {
    "architectures": ["BaselineClassifierForImageClassification"],
    "model_type": "baseline_classifier",
    "auto_map": {
        "AutoConfig": "hf_model.BaselineClassifierConfig",
        "AutoModel": "hf_model.BaselineClassifierForImageClassification",
    },
    **model_config,
}

config_path = hf_model_dir / "config.json"
with open(config_path, "w") as f:
    json.dump(config_dict, f, indent=2)

print(f"✓ Saved config.json")
print(json.dumps(config_dict, indent=2))

In [ ]:
# Configuration for your model
model_config = {
    "arch": "resnet18",           # Match your architecture
    "num_classes": 8,              # Match your num_classes
    "dropout_p": 0.5,              # Match your dropout
    "pretrained": False,           # No need to re-pretrain
}

# Create output directory
hf_model_dir = Path("./hf_checkpoint")
hf_model_dir.mkdir(parents=True, exist_ok=True)

print(f"Created directory: {hf_model_dir}")
print(f"Model config: {model_config}")

## Step 3: Create HuggingFace Model Directory Structure

In [ ]:
def extract_state_dict(checkpoint, device):
    """Extract and remap state dict from Lightning checkpoint."""
    state_dict = checkpoint['state_dict']
    
    # Remove 'net.' prefix (Lightning module wrapper)
    remapped_state_dict = {}
    for key, value in state_dict.items():
        # Remove 'net.' prefix
        new_key = key.replace('net.', '', 1)
        remapped_state_dict[new_key] = value
    
    return remapped_state_dict

# Extract the clean state dict
clean_state_dict = extract_state_dict(checkpoint, device)
print(f"Remapped state dict keys (first 5):\n{list(clean_state_dict.keys())[:5]}")

## Step 2: Extract Model Parameters from Checkpoint

The Lightning checkpoint has keys like `net.feature_extractor.layer1...` because they're wrapped. We need to remove the `net.` prefix.

In [ ]:
# Inspect state dict structure
state_dict = checkpoint['state_dict']
print(f"Number of parameters: {len(state_dict)}")
print(f"First 5 keys:\n{list(state_dict.keys())[:5]}")

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Point to your checkpoint
ckpt_path = "logs/train/runs/2026-01-29_13-44-38/checkpoints/last.ckpt"
checkpoint = torch.load(ckpt_path, map_location=device, weights_only=False)
print(f"Checkpoint keys: {checkpoint.keys()}")

## Step 1: Setup - Load Your Current Checkpoint

In [ ]:
import os
os.chdir("../")

import sys
import torch
import json
from pathlib import Path

# HuggingFace Model Migration

This notebook shows how to migrate your Lightning checkpoint to a HuggingFace-compatible model format that decouples code from checkpoints.

## Why Migrate?

1. **Code-Checkpoint Decoupling**: Load models without maintaining source code
2. **Easy Sharing**: Push to HuggingFace Hub with one command
3. **Standard API**: Use `AutoModel.from_pretrained()` just like any other HF model
4. **Future-Proof**: Update code without affecting model loading